In [1]:
# Import Libraries - 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

In [2]:
# data loading 
df = pd.read_csv("credit_card_fraud_balanced.csv")
df.head()

,amount,transaction_hour,merchant_category,card_type,transaction_channel,location,distance_from_home_km,prev_transactions_24hr,card_age_months,customer_age,credit_score,account_status,is_fraud
0,14.44,11,Dining,Mastercard,Contactless,Ahmedabad,0.67,5,80,26,637,Active,0
1,185.75,12,Grocery,Mastercard,POS,Bengaluru,4.28,7,61,49,668,Active,0
2,245.32,7,Grocery,Amex,POS,Kolkata,7.74,4,86,38,628,New,0
3,304.05,9,Grocery,Mastercard,Online,Kolkata,10.00,3,62,30,676,Active,0
4,255.64,16,Healthcare,Mastercard,Online,Foreign,17.64,1,24,31,566,Active,0


In [3]:
# drop the duplicated values 
df.drop_duplicates(inplace=True)

In [4]:
df.head()

,amount,transaction_hour,merchant_category,card_type,transaction_channel,location,distance_from_home_km,prev_transactions_24hr,card_age_months,customer_age,credit_score,account_status,is_fraud
0,14.44,11,Dining,Mastercard,Contactless,Ahmedabad,0.67,5,80,26,637,Active,0
1,185.75,12,Grocery,Mastercard,POS,Bengaluru,4.28,7,61,49,668,Active,0
2,245.32,7,Grocery,Amex,POS,Kolkata,7.74,4,86,38,628,New,0
3,304.05,9,Grocery,Mastercard,Online,Kolkata,10.00,3,62,30,676,Active,0
4,255.64,16,Healthcare,Mastercard,Online,Foreign,17.64,1,24,31,566,Active,0


In [5]:
df.shape

(6230, 13)

In [6]:
# Value counts of categorcial columns
df["merchant_category"].value_counts(),df["card_type"].value_counts(),df["transaction_channel"].value_counts,df["location"].value_counts(),df["account_status"].value_counts()

(merchant_category
 Grocery          1150
 Dining            880
 Travel            695
 Electronics       678
 Online            668
 Entertainment     605
 Clothing          550
 Healthcare        450
 Fuel              254
 Utilities         157
 Jewelry           114
 Gaming             29
 Name: count, dtype: int64,
 card_type
 Visa          2332
 Mastercard    2115
 RuPay         1122
 Amex           661
 Name: count, dtype: int64,
 <bound method IndexOpsMixin.value_counts of 0       Contactless
 1               POS
 2               POS
 3            Online
 4            Online
            ...     
 6225    Contactless
 6226         Online
 6227    Contactless
 6228         Mobile
 6229         Mobile
 Name: transaction_channel, Length: 6230, dtype: object>,
 location
 Mumbai        1092
 Delhi          942
 Foreign        867
 Bengaluru      688
 Hyderabad      514
 Chennai        511
 Pune           483
 Kolkata        385
 Ahmedabad      329
 Jaipur         237
 Chandigarh    

In [7]:
# handling categorical data 
label = LabelEncoder()

# 1. merchant_category
df["merchant_category"] = df["merchant_category"].map({
    'Clothing': 0, 'Dining': 1, 'Electronics': 2, 'Entertainment': 3,
    'Fuel': 4, 'Gaming': 5, 'Grocery': 6, 'Healthcare': 7,
    'Jewelry': 8, 'Online': 9, 'Travel': 10, 'Utilities': 11
})

# 2. card_type
df["card_type"] = df["card_type"].map({
    "Visa": 0, "Mastercard": 1, "RuPay": 2, "Amex": 3
})

# 3. transaction_channel - Yahi 768 ka problem tha
df["transaction_channel"] = label.fit_transform(df["transaction_channel"])
# 4. location - 
df["location"] = label.fit_transform(df["location"])

# 5. account_status
df["account_status"] = df["account_status"].map({
    "Active": 0, "New": 1, "Premium": 2, "Suspended": 3, "Flagged": 4
})

# Final check
print(df.isnull().sum())

amount                    0
transaction_hour          0
merchant_category         0
card_type                 0
transaction_channel       0
location                  0
distance_from_home_km     0
prev_transactions_24hr    0
card_age_months           0
customer_age              0
credit_score              0
account_status            0
is_fraud                  0
dtype: int64


In [8]:
# feature selection 
x=df.drop(columns=["is_fraud"])
y=df["is_fraud"]

In [9]:
df.head()

,amount,transaction_hour,merchant_category,card_type,transaction_channel,location,distance_from_home_km,prev_transactions_24hr,card_age_months,customer_age,credit_score,account_status,is_fraud
0,14.44,11,1,1,1,0,0.67,5,80,26,637,0,0
1,185.75,12,6,1,4,1,4.28,7,61,49,668,0,0
2,245.32,7,6,3,4,8,7.74,4,86,38,628,1,0
3,304.05,9,6,1,3,8,10.00,3,62,30,676,0,0
4,255.64,16,7,1,3,5,17.64,1,24,31,566,0,0


In [10]:
# data scaling 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [11]:
# data scaling 
scaler=StandardScaler()
x_train_scale=scaler.fit_transform(x_train)
x_test_scale=scaler.transform(x_test)

In [12]:
x_test_scale

array([[-0.20714927,  0.5958723 , -1.1513792 , ..., -1.24774878,
         1.73318064, -0.64134427],
       [-0.24523843,  0.06246617, -0.28032973, ...,  1.40747367,
         0.70051515, -0.64134427],
       [-0.2253679 ,  1.30708048,  1.1714194 , ..., -0.74989457,
        -1.42869205,  0.34006803],
       ...,
       [-0.25503653,  0.41807026, -0.28032973, ...,  0.32878955,
         1.85028704, -0.64134427],
       [-0.255287  ,  0.5958723 , -1.1513792 , ...,  0.57771666,
         1.4244456 ,  1.32148032],
       [-0.25321765, -0.47093997, -1.1513792 , ..., -1.66262729,
         1.2754011 , -0.64134427]], shape=(1246, 12))

In [13]:
x_train_scale

array([[-0.25822106, -0.29313792,  1.46176922, ..., -0.00311325,
         0.92408191, -0.64134427],
       [-0.14418617, -0.82654406, -1.1513792 , ..., -0.83287027,
        -0.06599944, -0.64134427],
       [-0.22159297,  0.06246617,  1.46176922, ..., -1.08179737,
        -1.79065728,  0.34006803],
       ...,
       [-0.24018134, -1.18214815,  1.46176922, ...,  1.15854657,
        -0.9389744 ,  0.34006803],
       [-0.19736309, -0.47093997,  0.0100201 , ...,  0.57771666,
        -2.20585268, -0.64134427],
       [-0.14855745,  1.48488253, -0.86102938, ..., -1.16477308,
         0.14692128, -0.64134427]], shape=(4984, 12))

In [14]:
# paramter for random forest tunning 
para_random={"max_depth":[4,6,8],"n_estimators":[100,150,200]}

In [15]:
tunning_model=RandomForestClassifier(random_state=42)

In [16]:
# model using Random forest
# model selction (Initilaize)
grid_search=GridSearchCV(estimator=tunning_model,
                         param_grid=para_random,
                        cv=5,
                        n_jobs=-1,
                        scoring="accuracy",
                        verbose=1)    # for classfication scoring will be accuracy_score
# model traning 
grid_search.fit(x_train_scale,y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [4, 6, ...], 'n_estimators': [100, 150, ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [17]:
# check the best combination 
grid_search.best_params_

{'max_depth': 8, 'n_estimators': 200}

In [18]:
model=grid_search.best_estimator_

In [19]:
# predict 
y_pred=model.predict(x_test_scale)
accuracy_score(y_test,y_pred)

0.9879614767255217

In [20]:
# traning accuracy 
y_pred_train=model.predict(x_train_scale)
accuracy_score(y_train,y_pred_train)

0.9979935794542536

# PREDECTIVE MODEL 


In [22]:
# amount	transaction_hour	transaction_hour	card_type	transaction_channel	location	distance_from_home_km	prev_transactions_24hr	card_age_months	customer_age	credit_score	account_status	encoded_merchant_category	encoded_locatioN
# PREDECTIVE MODEL - FINAL

amount=int(input("Enter Amount : "))
hour=int(input("Enter Transaction hour : "))
card=int(input("Enter card type ---> Visa:0, Mastercard:1, RuPay:2, Amex:3 : "))
merchant=int(input("Enter Merchant category ---> Dining: 1, 'Grocery': 6, 'Healthcare': 7 : "))
channel=int(input("Enter Transaction Channel ---> Online:0,POS:1,Contactless:2,ATM:3,Mobile:4: "))
location=int(input("Enter Location ---> 'Ahmedabad': 0, 'Bengaluru': 1, 'Kolkata': 8, 'Foreign': 5, 'Pune' : "))
distance=int(input("Enter distance_from_home :"))
prev=int(input("Enter prev_transactions_24hr : "))
age=int(input("Enter card age month: "))
cust_age=int(input("Enter Customer age : "))
score=int(input("Enter credit_score : "))
status=int(input("Enter Account status ----> Active:0, New:1, Premium:2, Suspended:3, Flagged:4 : "))

# Scaler data - 
import pandas as pd
data=pd.DataFrame([[amount,hour,card,merchant,channel,location,distance,prev,age,cust_age,score,status]], columns=x.columns)
data = scaler.transform(data)

pred=model.predict(data)
print("Fraud : ",pred)

if pred[0]==1:
    print("Result: FRAUD Transaction")
else:
    print("Result: Genuine Transaction")

Enter Amount :  1000
Enter Transaction hour :  14
Enter card type ---> Visa:0, Mastercard:1, RuPay:2, Amex:3 :  1
Enter Merchant category ---> Dining: 1, 'Grocery': 6, 'Healthcare': 7 :  1
Enter Transaction Channel ---> Online:0,POS:1,Contactless:2,ATM:3,Mobile:4:  0
Enter Location ---> 'Ahmedabad': 0, 'Bengaluru': 1, 'Kolkata': 8, 'Foreign': 5, 'Pune' :  1
Enter distance_from_home : 100
Enter prev_transactions_24hr :  2
Enter card age month:  24
Enter Customer age :  35
Enter credit_score :  700
Enter Account status ----> Active:0, New:1, Premium:2, Suspended:3, Flagged:4 :  0


Fraud :  [1]
Result: FRAUD Transaction


In [23]:
# amount	transaction_hour	card_type	transaction_channel	location	distance_from_home_km	prev_transactions_24hr	card_age_months	customer_age	credit_score	account_status	
# PREDECTIVE MODEL - FINAL 2 for (GENIUN)

amount=int(input("Enter Amount : "))
hour=int(input("Enter Transaction hour : "))
card=int(input("Enter card type ---> Visa:0, Mastercard:1, RuPay:2, Amex:3 : "))
merchant=int(input("Enter Merchant category ---> Dining: 1, 'Grocery': 6, 'Healthcare': 7 : "))
channel=int(input("Enter Transaction Channel ---> Online:0,POS:1,Contactless:2,ATM:3,Mobile:4: "))
location=int(input("Enter Location ---> 'Ahmedabad': 0, 'Bengaluru': 1, 'Kolkata': 8, 'Foreign': 5, 'Pune' : "))
distance=int(input("Enter distance_from_home :"))
prev=int(input("Enter prev_transactions_24hr : "))
age=int(input("Enter card age month: "))
cust_age=int(input("Enter Customer age : "))
score=int(input("Enter credit_score : "))
status=int(input("Enter Account status ----> Active:0, New:1, Premium:2, Suspended:3, Flagged:4 : "))

# Scaler data - 
import pandas as pd
data=pd.DataFrame([[amount,hour,card,merchant,channel,location,distance,prev,age,cust_age,score,status]], columns=x.columns)
data = scaler.transform(data)

pred=model.predict(data)
print("Fraud : ",pred)

if pred[0]==1:
    print("Result: FRAUD Transaction")
else:
    print("Result: Genuine Transaction")

Enter Amount :  5000
Enter Transaction hour :  10
Enter card type ---> Visa:0, Mastercard:1, RuPay:2, Amex:3 :  0
Enter Merchant category ---> Dining: 1, 'Grocery': 6, 'Healthcare': 7 :  0
Enter Transaction Channel ---> Online:0,POS:1,Contactless:2,ATM:3,Mobile:4:  1
Enter Location ---> 'Ahmedabad': 0, 'Bengaluru': 1, 'Kolkata': 8, 'Foreign': 5, 'Pune' :  0
Enter distance_from_home : 5
Enter prev_transactions_24hr :  1
Enter card age month:  36
Enter Customer age :  40
Enter credit_score :  800
Enter Account status ----> Active:0, New:1, Premium:2, Suspended:3, Flagged:4 :  2


Fraud :  [0]
Result: Genuine Transaction


In [25]:
import joblib
joblib.dump(model, "fraud_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Saved!")

Saved!
